In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

REAL_ROOT = "/content/drive/MyDrive/ff_frames_fixed/real"
FAKE_ROOT = "/content/drive/MyDrive/ff_frames_fixed/fake"

print(f"Real exists: {os.path.exists(REAL_ROOT)}")
print(f"Fake exists: {os.path.exists(FAKE_ROOT)}")

if os.path.exists(REAL_ROOT) and os.path.exists(FAKE_ROOT):
    print(f"\nReal folders: {len(os.listdir(REAL_ROOT))}")
    print(f"Fake folders: {len(os.listdir(FAKE_ROOT))}")
    print("\n✅ Both folders found! Ready to continue.")
else:
    print("\n❌ Still missing folders — share output here")

Real exists: True
Fake exists: True

Real folders: 1031
Fake folders: 1031

✅ Both folders found! Ready to continue.


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

def get_valid_folders(root):
    folders = []
    for folder in os.listdir(root):
        path = os.path.join(root, folder)
        if os.path.isdir(path) and len(os.listdir(path)) >= 20:
            folders.append(path)
    return folders

real_folders = get_valid_folders(REAL_ROOT)
fake_folders = get_valid_folders(FAKE_ROOT)

real_train, real_temp = train_test_split(real_folders, test_size=0.2, random_state=42)
real_val, real_test   = train_test_split(real_temp,    test_size=0.5, random_state=42)

fake_train, fake_temp = train_test_split(fake_folders, test_size=0.2, random_state=42)
fake_val, fake_test   = train_test_split(fake_temp,    test_size=0.5, random_state=42)

def make_df(real_list, fake_list):
    data = (
        [{"path": p, "label": 0} for p in real_list] +
        [{"path": p, "label": 1} for p in fake_list]
    )
    return pd.DataFrame(data)

train_df = make_df(real_train, fake_train)
val_df   = make_df(real_val,   fake_val)
test_df  = make_df(real_test,  fake_test)

train_df.to_csv("/content/drive/MyDrive/train_split.csv", index=False)
val_df.to_csv("/content/drive/MyDrive/val_split.csv",     index=False)
test_df.to_csv("/content/drive/MyDrive/test_split.csv",   index=False)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
print("✅ CSVs saved to Drive!")

Train: 1647 | Val: 206 | Test: 206
✅ CSVs saved to Drive!


In [ ]:
import torch
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

MAX_FRAMES_PER_VIDEO = 15

train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

val_test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

class DeepfakeDataset(Dataset):
    def __init__(self, df, transform=None, max_frames=MAX_FRAMES_PER_VIDEO):
        self.transform = transform
        self.samples   = []

        for _, row in df.iterrows():
            folder = row["path"]
            label  = row["label"]
            if not os.path.isdir(folder):
                continue

            all_frames = sorted([
                os.path.join(folder, f)
                for f in os.listdir(folder)
                if f.lower().endswith((".jpg", ".jpeg", ".png"))
            ])

            if len(all_frames) > max_frames:
                indices    = np.linspace(0, len(all_frames)-1,
                                         max_frames, dtype=int)
                all_frames = [all_frames[i] for i in indices]

            for frame_path in all_frames:
                self.samples.append((frame_path, label))

        print(f"Total frames loaded: {len(self.samples)}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = Image.open(path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, torch.tensor(label, dtype=torch.float32)

train_dataset = DeepfakeDataset(train_df, transform=train_transforms)
val_dataset   = DeepfakeDataset(val_df,   transform=val_test_transforms)
test_dataset  = DeepfakeDataset(test_df,  transform=val_test_transforms)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f"\nMAX_FRAMES_PER_VIDEO : {MAX_FRAMES_PER_VIDEO}")
print(f"Train batches        : {len(train_loader)}")
print(f"Val batches          : {len(val_loader)}")
print(f"Test batches         : {len(test_loader)}")

Total frames loaded: 24705
Total frames loaded: 3090
Total frames loaded: 3090

MAX_FRAMES_PER_VIDEO : 15
Train batches        : 773
Val batches          : 97
Test batches         : 97


In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ── Load EfficientNet B0 ──────────────────────────────────
model = models.efficientnet_b0(weights="IMAGENET1K_V1")

# ── Freeze all layers ─────────────────────────────────────
for param in model.parameters():
    param.requires_grad = False

# ── Unfreeze last 2 blocks + classifier ───────────────────
for param in model.features[6].parameters():
    param.requires_grad = True
for param in model.features[7].parameters():
    param.requires_grad = True
for param in model.features[8].parameters():
    param.requires_grad = True

# ── Replace classifier head ───────────────────────────────
in_features = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(p=0.3),   # slightly less dropout for smaller model
    nn.Linear(in_features, 1)
)

model = model.to(device)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params     : {total_params:,}")
print(f"Trainable params : {trainable_params:,}")

Using device: cuda
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 118MB/s] 


Total params     : 4,008,829
Trainable params : 3,157,021


In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

criterion = nn.BCEWithLogitsLoss()
optimizer = AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4,
    weight_decay=1e-2
)
scheduler = CosineAnnealingLR(optimizer, T_max=5)

print("✅ Loss, optimizer and scheduler ready!")

✅ Loss, optimizer and scheduler ready!


In [ ]:
import time
from sklearn.metrics import roc_auc_score
import numpy as np

NUM_EPOCHS       = 10
PATIENCE         = 3
best_val_loss    = float("inf")
patience_counter = 0
history          = {"train_loss": [], "val_loss": [], "val_acc": [], "val_auc": []}

print(f"Training on {device} for up to {NUM_EPOCHS} epochs (early stop patience={PATIENCE})\n")

for epoch in range(NUM_EPOCHS):
    start = time.time()

    # ── Training ─────────────────────────────────────────
    model.train()
    train_loss    = 0.0
    train_correct = 0
    train_total   = 0

    for batch_idx, (images, labels) in enumerate(train_loader):
        images = images.to(device)
        labels = labels.to(device).unsqueeze(1)

        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss    += loss.item()
        preds          = (torch.sigmoid(outputs) >= 0.5).float()
        train_correct += (preds == labels).sum().item()
        train_total   += labels.size(0)

        if (batch_idx + 1) % 200 == 0:
            print(f"  Batch [{batch_idx+1}/{len(train_loader)}] "
                  f"Loss: {train_loss/(batch_idx+1):.4f}")

    train_loss /= len(train_loader)
    train_acc   = train_correct / train_total * 100

    # ── Validation ───────────────────────────────────────
    model.eval()
    val_loss    = 0.0
    val_correct = 0
    val_total   = 0
    all_probs   = []
    all_labels  = []

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device).unsqueeze(1)

            outputs   = model(images)
            loss      = criterion(outputs, labels)
            val_loss += loss.item()

            probs     = torch.sigmoid(outputs)
            preds     = (probs >= 0.5).float()
            val_correct += (preds == labels).sum().item()
            val_total   += labels.size(0)

            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    val_loss /= len(val_loader)
    val_acc   = val_correct / val_total * 100
    val_auc   = roc_auc_score(all_labels, all_probs)
    elapsed   = (time.time() - start) / 60

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    history["val_auc"].append(val_auc)

    print(f"\nEpoch [{epoch+1}/{NUM_EPOCHS}]")
    print(f"  Train Loss : {train_loss:.4f} | Train Acc : {train_acc:.2f}%")
    print(f"  Val Loss   : {val_loss:.4f} | Val Acc   : {val_acc:.2f}% | AUC: {val_auc:.4f}")
    print(f"  Time       : {elapsed:.1f} min")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save(model.state_dict(),
                   "/content/drive/MyDrive/efficientnet_deepfake_best.pth")
        print(f"  ✅ Best model saved (Val Loss: {val_loss:.4f} | AUC: {val_auc:.4f})")
    else:
        patience_counter += 1
        print(f"  ⚠️  No improvement ({patience_counter}/{PATIENCE})")
        if patience_counter >= PATIENCE:
            print("\n🛑 Early stopping triggered!")
            break

    scheduler.step()
    print("-" * 60)

print(f"\n{'='*60}")
print(f"Training finished!")
print(f"Best Val Loss : {best_val_loss:.4f}")
print(f"Best Val AUC  : {max(history['val_auc']):.4f}")
print(f"Best Val Acc  : {max(history['val_acc']):.2f}%")
print(f"{'='*60}")

Training on cuda for up to 10 epochs (early stop patience=3)

  Batch [200/773] Loss: 0.5639
  Batch [400/773] Loss: 0.4937
  Batch [600/773] Loss: 0.4453

Epoch [1/10]
  Train Loss : 0.4158 | Train Acc : 79.59%
  Val Loss   : 0.3833 | Val Acc   : 82.27% | AUC: 0.9085
  Time       : 72.7 min
  ✅ Best model saved (Val Loss: 0.3833 | AUC: 0.9085)
------------------------------------------------------------
  Batch [200/773] Loss: 0.2664
  Batch [400/773] Loss: 0.2544
  Batch [600/773] Loss: 0.2510

Epoch [2/10]
  Train Loss : 0.2440 | Train Acc : 89.69%
  Val Loss   : 0.3820 | Val Acc   : 83.66% | AUC: 0.9243
  Time       : 2.7 min
  ✅ Best model saved (Val Loss: 0.3820 | AUC: 0.9243)
------------------------------------------------------------
  Batch [200/773] Loss: 0.1817
  Batch [400/773] Loss: 0.1810
  Batch [600/773] Loss: 0.1862

Epoch [3/10]
  Train Loss : 0.1840 | Train Acc : 92.16%
  Val Loss   : 0.3940 | Val Acc   : 85.02% | AUC: 0.9331
  Time       : 2.7 min
  ⚠️  No improvem

In [ ]:
from sklearn.metrics import (roc_auc_score, f1_score,
                              confusion_matrix, classification_report)
import numpy as np
import torch

# ── Load best saved model ─────────────────────────────────
model.load_state_dict(torch.load(
    "/content/drive/MyDrive/efficientnet_deepfake_best.pth",
    map_location=device))
model.eval()

test_loss    = 0.0
test_correct = 0
test_total   = 0
all_probs    = []
all_labels   = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device).unsqueeze(1)

        outputs   = model(images)
        loss      = criterion(outputs, labels)
        test_loss += loss.item()

        probs     = torch.sigmoid(outputs)
        preds     = (probs >= 0.35).float()
        test_correct += (preds == labels).sum().item()
        test_total   += labels.size(0)

        all_probs.extend(probs.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

test_loss /= len(test_loader)
test_acc   = test_correct / test_total * 100
test_auc   = roc_auc_score(all_labels, all_probs)
all_preds  = (np.array(all_probs) >= 0.35).astype(int)

print(f"\n{'='*60}")
print(f"TEST SET RESULTS")
print(f"{'='*60}")
print(f"Test Loss     : {test_loss:.4f}")
print(f"Test Accuracy : {test_acc:.2f}%")
print(f"Test AUC-ROC  : {test_auc:.4f}")
print(f"Test F1 Score : {f1_score(all_labels, all_preds):.4f}")
print(f"\nConfusion Matrix:")
print(confusion_matrix(all_labels, all_preds))
print(f"\nClassification Report:")
print(classification_report(all_labels, all_preds,
                             target_names=["Real", "Fake"]))


TEST SET RESULTS
Test Loss     : 0.3962
Test Accuracy : 82.46%
Test AUC-ROC  : 0.9181
Test F1 Score : 0.8203

Confusion Matrix:
[[1311  234]
 [ 308 1237]]

Classification Report:
              precision    recall  f1-score   support

        Real       0.81      0.85      0.83      1545
        Fake       0.84      0.80      0.82      1545

    accuracy                           0.82      3090
   macro avg       0.83      0.82      0.82      3090
weighted avg       0.83      0.82      0.82      3090



In [ ]:
import numpy as np
from sklearn.metrics import (roc_auc_score, f1_score,
                              confusion_matrix, classification_report)

# Try different thresholds
thresholds = [0.5, 0.45, 0.4, 0.35, 0.3]

print(f"{'Threshold':<12} {'Accuracy':<12} {'Fake Recall':<14} {'Real Recall':<14} {'F1':<8}")
print("-" * 60)

for thresh in thresholds:
    preds = (np.array(all_probs) >= thresh).astype(int)

    cm           = confusion_matrix(all_labels, preds)
    tn, fp, fn, tp = cm.ravel()

    acc          = (tp + tn) / (tp + tn + fp + fn) * 100
    fake_recall  = tp / (tp + fn) * 100
    real_recall  = tn / (tn + fp) * 100
    f1           = f1_score(all_labels, preds)

    print(f"{thresh:<12} {acc:<12.2f} {fake_recall:<14.2f} {real_recall:<14.2f} {f1:<8.4f}")

Threshold    Accuracy     Fake Recall    Real Recall    F1      
------------------------------------------------------------
0.5          82.46        80.06          84.85          0.8203  
0.45         82.62        82.39          82.85          0.8258  
0.4          82.94        84.47          81.42          0.8320  
0.35         83.37        86.80          79.94          0.8392  
0.3          83.40        88.41          78.38          0.8419  
